In [ ]:
# Only run this if importing fresh data from StatsBomb. Otherwise skip to the checkpoint load cell in the other notebook.

from statsbombpy import sb
import pandas as pd
import numpy as np
import os

os.makedirs('data', exist_ok=True)

# 7 competition-seasons across 5 competitions
OUR_COMPS = [
    (55,  43, "UEFA Euro",         "2020"),
    (55, 282, "UEFA Euro",         "2024"),
    (43, 106, "FIFA World Cup",    "2022"),
    (53, 106, "UEFA Women's Euro", "2022"),
    (53, 315, "UEFA Women's Euro", "2025"),
    (72, 107, "Women's World Cup", "2023"),
    (9,  281, "1. Bundesliga",     "2023/24"),
]

for c_id, s_id, name, season in OUR_COMPS:
    matches = sb.matches(competition_id=c_id, season_id=s_id)
    n = (matches['match_status_360'] == 'available').sum() \
        if 'match_status_360' in matches.columns else len(matches)
    print(f"{name} {season}: {n} matches")

# Load shots with 360 freeze-frame data
all_shots = []

for c_id, s_id, comp_name, season in OUR_COMPS:
    matches = sb.matches(competition_id=c_id, season_id=s_id)

    if 'match_status_360' in matches.columns:
        m_ids = matches[matches['match_status_360'] == 'available']['match_id'].tolist()
    else:
        m_ids = matches['match_id'].tolist()

    for m_id in m_ids:
        try:
            events = sb.events(match_id=m_id)
            shots = events[events['type'] == 'Shot'].copy()

            if len(shots) == 0:
                continue

            match_row = matches[matches['match_id'] == m_id].iloc[0]
            shots['match_id'] = m_id
            shots['competition_name'] = comp_name
            shots['competition_id'] = c_id
            shots['season_id'] = s_id
            shots['season_name'] = season
            shots['home_team'] = match_row['home_team']
            shots['away_team'] = match_row['away_team']
            shots['match_date'] = match_row['match_date']

            shots = shots[shots['shot_freeze_frame'].notna()].copy()

            if len(shots) > 0:
                all_shots.append(shots)

        except Exception as e:
            print(f"Match {m_id} failed: {e}")
            continue

df_raw = pd.concat(all_shots, ignore_index=True)
df_raw['is_goal'] = (df_raw['shot_outcome'] == 'Goal').astype(int)

df_raw = df_raw[df_raw['shot_type'] != 'Penalty'].copy()
df_raw = df_raw.reset_index(drop=True)

print(f"Total shots: {len(df_raw):,} | Goals: {df_raw['is_goal'].sum():,} "
      f"({df_raw['is_goal'].mean()*100:.2f}%) | Matches: {df_raw['match_id'].nunique():,}")

comp_summary = df_raw.groupby(['competition_name', 'season_name']).agg(
    Matches=('match_id', 'nunique'),
    Shots=('id', 'count'),
    Goals=('is_goal', 'sum'),
    Conv=('is_goal', 'mean'),
).reset_index()
comp_summary['Conv%'] = (comp_summary['Conv'] * 100).round(2)
print(comp_summary[['competition_name', 'season_name', 'Matches', 'Shots', 'Goals', 'Conv%']]
      .sort_values('Shots', ascending=False).to_string(index=False))

In [ ]:
# Spatial and tracking feature extraction from freeze-frame data.

GOAL_X = 120
GOAL_Y_CENTRE = 40
GOAL_Y_LEFT = 36
GOAL_Y_RIGHT = 44
GOAL_CENTRE = np.array([GOAL_X, GOAL_Y_CENTRE])

def is_goalkeeper(player):
    """Detect GK from position field. Handles both old (keeper bool) and
    new (position dict) StatsBomb structures."""
    pos = player.get('position')
    if isinstance(pos, dict):
        return pos.get('name') == 'Goalkeeper'
    return bool(player.get('keeper', False))

def get_richer_features(row):
    shot_loc = np.array(row['location'])
    x, y = shot_loc[0], shot_loc[1]
    freeze = row.get('shot_freeze_frame')

    distance = np.sqrt((120 - x)**2 + (40 - y)**2)

    v1_x = 120 - x; v1_y = 36 - y
    v2_x = 120 - x; v2_y = 44 - y
    dot = v1_x*v2_x + v1_y*v2_y
    angle_rad = np.arccos(np.clip(
        dot / (np.sqrt(v1_x**2 + v1_y**2) * np.sqrt(v2_x**2 + v2_y**2)),
        -1, 1
    ))

    y_centrality = abs(y - 40)
    near_post_dist = np.sqrt((120-x)**2 + (36-y)**2)
    far_post_dist = np.sqrt((120-x)**2 + (44-y)**2)

    body_part = str(row.get('shot_body_part', ''))
    is_header = int(body_part == 'Head')
    body_part_prob = (0.9 if body_part in ['Right Foot', 'Left Foot']
                      else 0.6 if body_part == 'Head' else 0.3)

    under_pressure = int(bool(row.get('under_pressure')))
    shot_first_time = int(bool(row.get('shot_first_time')))
    shot_one_on_one = int(bool(row.get('shot_one_on_one')))
    shot_open_goal = int(bool(row.get('shot_open_goal')))

    technique = str(row.get('shot_technique', ''))
    play_pat = str(row.get('play_pattern', ''))
    aerial_set_piece = int(is_header and play_pat in ['From Corner', 'From Free Kick'])
    is_diving_header = int('Diving' in technique)

    base = dict(
        distance=distance,
        angle_rad=angle_rad,
        y_centrality=y_centrality,
        near_post_dist=near_post_dist,
        far_post_dist=far_post_dist,
        body_part_prob=body_part_prob,
        is_header=is_header,
        under_pressure=under_pressure,
        shot_first_time=shot_first_time,
        shot_one_on_one=shot_one_on_one,
        shot_open_goal=shot_open_goal,
        aerial_set_piece=aerial_set_piece,
        is_diving_header=is_diving_header,
    )

    ff_defaults = dict(
        num_opponents=0,
        cone_opponents=0,
        gk_angle_error=0.0,
        gk_dist=0.0,
        gk_off_line=0.0,
        nearest_opp_dist=10.0,
        nearest_block_dist=5.0,
        defenders_in_box=0,
        attackers_ahead=0,
        cone_advantage=0,
        gk_lateral_position=0.0,
        defender_in_shot_path=0,
        n_attackers_in_box=0,
    )

    if not isinstance(freeze, list) or len(freeze) == 0:
        return pd.Series({**base, **ff_defaults})

    gk = next((p for p in freeze if is_goalkeeper(p)), None)
    opponents = [p for p in freeze if not p.get('teammate') and not is_goalkeeper(p)]
    teammates = [p for p in freeze if p.get('teammate') and not is_goalkeeper(p)]

    def is_in_cone(p_loc):
        p_x, p_y = p_loc[0], p_loc[1]
        cp1 = (120 - x) * (p_y - y) - (36 - y) * (p_x - x)
        cp2 = (120 - x) * (p_y - y) - (44 - y) * (p_x - x)
        return (cp1 * cp2 < 0) and (p_x > x)

    cone_opps = len([p for p in opponents if is_in_cone(p['location'])])
    cone_teammates = len([p for p in teammates if is_in_cone(p['location'])])
    cone_advantage = cone_teammates - cone_opps

    if gk:
        gk_loc = np.array(gk['location'])
        gk_dist = float(np.linalg.norm(gk_loc - shot_loc))
        gk_off_line = float(120 - gk_loc[0])

        v_goal = np.array([120, 40]) - shot_loc
        v_gk = gk_loc - shot_loc
        gk_angle_error = float(abs(np.arccos(np.clip(
            np.dot(v_goal, v_gk) / (np.linalg.norm(v_goal) * np.linalg.norm(v_gk) + 1e-9),
            -1.0, 1.0
        ))))
        gk_lateral_position = float(abs(gk_loc[1] - 40))
    else:
        gk_dist = gk_off_line = gk_angle_error = gk_lateral_position = 0.0

    def dist_to_shot_line(p_loc):
        p = np.array(p_loc) - shot_loc
        goal = np.array([120, 40]) - shot_loc
        goal_norm = goal / (np.linalg.norm(goal) + 1e-9)
        return float(np.linalg.norm(p - np.dot(p, goal_norm) * goal_norm))

    opp_locs = [p['location'] for p in opponents]

    nearest_opp_dist = (min([np.linalg.norm(np.array(loc) - shot_loc) for loc in opp_locs])
                         if opp_locs else 10.0)
    nearest_block_dist = (min([dist_to_shot_line(loc) for loc in opp_locs])
                           if opp_locs else 5.0)
    defenders_in_box = len([p for p in opponents if p['location'][0] > 102])
    n_attackers_in_box = len([p for p in teammates if p['location'][0] > 102])
    attackers_ahead = len([p for p in teammates if p['location'][0] > x])

    shot_vec = GOAL_CENTRE - shot_loc
    shot_vec_norm = shot_vec / (np.linalg.norm(shot_vec) + 1e-9)
    defender_in_shot_path = 0
    for opp in opponents:
        opp_vec = np.array(opp['location']) - shot_loc
        proj = float(np.dot(opp_vec, shot_vec_norm))
        if proj > 0:
            perp = float(np.linalg.norm(opp_vec - proj * shot_vec_norm))
            if perp < 1.5 and proj < float(np.linalg.norm(shot_vec)):
                defender_in_shot_path = 1
                break

    return pd.Series({
        **base,
        'num_opponents': len(opponents),
        'cone_opponents': cone_opps,
        'gk_angle_error': gk_angle_error,
        'gk_dist': gk_dist,
        'gk_off_line': gk_off_line,
        'nearest_opp_dist': nearest_opp_dist,
        'nearest_block_dist': nearest_block_dist,
        'defenders_in_box': defenders_in_box,
        'attackers_ahead': attackers_ahead,
        'cone_advantage': cone_advantage,
        'gk_lateral_position': gk_lateral_position,
        'defender_in_shot_path': defender_in_shot_path,
        'n_attackers_in_box': n_attackers_in_box,
    })

spatial_features = df_raw.apply(get_richer_features, axis=1)

df_rich = df_raw.copy()
for col in spatial_features.columns:
    df_rich[col] = spatial_features[col].values

# body_part_prob derived from actual conversion rates rather than fixed constants
header_means = df_rich.groupby('is_header')['is_goal'].mean()
df_rich['body_part_prob'] = df_rich['is_header'].map(header_means)

# Interaction terms
df_rich['gk_dist_x_off_line'] = df_rich['gk_dist'] * df_rich['gk_off_line']
df_rich['nearest_opp_x_cone'] = df_rich['nearest_opp_dist'] * df_rich['cone_opponents']
df_rich['distance_x_defenders'] = df_rich['distance'] * df_rich['defenders_in_box']
df_rich['angle_x_cone_adv'] = df_rich['angle_rad'] * df_rich['cone_advantage']
df_rich['header_x_gk_off_line'] = df_rich['is_header'] * df_rich['gk_off_line']
df_rich['pressure_x_nearest_opp'] = df_rich['under_pressure'] * df_rich['nearest_opp_dist']

# Cap gk_off_line at 20m - a few implausible values above this
df_rich['gk_off_line'] = df_rich['gk_off_line'].clip(0, 20)

# Remove 'Other' play pattern (anomalous shots)
df_rich = df_rich[df_rich['play_pattern'] != 'Other'].copy()
df_rich = df_rich.reset_index(drop=True)

print(f"Shots: {len(df_rich):,} | Goals: {df_rich['is_goal'].sum():,} "
      f"({df_rich['is_goal'].mean()*100:.2f}%)")
print(f"GK detected in {(df_rich['gk_dist']>0).sum():,}/{len(df_rich):,} shots")

df_rich.to_pickle('data/df_rich_spatial_v3.pkl')

In [ ]:
# Temporal feature extraction: 10-second pre-shot window + possession chain.

def ts_to_seconds(ts):
    h, m, s = ts.split(':')
    return int(h)*3600 + int(m)*60 + float(s)

all_events = []
match_ids = df_rich['match_id'].unique()
failed = []

for mid in match_ids:
    try:
        ev = sb.events(match_id=mid)
        ev['match_id'] = mid
        ev['time_seconds'] = ev['timestamp'].apply(ts_to_seconds)
        all_events.append(ev)
    except Exception as e:
        print(f"Match {mid} failed: {e}")
        failed.append(mid)

events_df = pd.concat(all_events, ignore_index=True)
print(f"Events loaded: {len(events_df):,} across {events_df['match_id'].nunique():,} matches "
      f"({len(failed)} failed)")

events_df.to_pickle('data/events_df_v3.pkl')


def extract_temporal_features(shot_row, match_events):
    """Temporal features from the 10s window before the shot and the
    full possession chain leading to it."""
    shot_time = ts_to_seconds(shot_row['timestamp'])
    shot_period = shot_row['period']
    shot_poss = shot_row['possession']
    shot_loc = np.array(shot_row['location'])

    window = match_events[
        (match_events['period'] == shot_period) &
        (match_events['time_seconds'] >= shot_time - 10) &
        (match_events['time_seconds'] < shot_time)
    ]

    w_passes = int((window['type'] == 'Pass').sum())
    w_carries = int((window['type'] == 'Carry').sum())
    w_pressure_events = int((window['type'] == 'Pressure').sum())
    w_miscontrol = int((window['type'] == 'Miscontrol').sum())
    w_dribble = int((window['type'] == 'Dribble').sum())
    w_under_pressure = int(window['under_pressure'].fillna(False).sum())

    carries_w = window[window['type'] == 'Carry']
    if len(carries_w) > 0:
        carry_x = carries_w['location'].dropna().apply(lambda l: l[0])
        w_prog_distance = float(carry_x.max() - carry_x.min())
    else:
        w_prog_distance = 0.0

    w_duration = float(window['duration'].sum())
    w_speed = w_prog_distance / (w_duration + 1e-9)

    chain = match_events[match_events['possession'] == shot_poss].sort_values('index')

    chain_passes = int((chain['type'] == 'Pass').sum())
    chain_carries = int((chain['type'] == 'Carry').sum())
    chain_duration = float(chain['duration'].sum())
    chain_pressures = int((chain['type'] == 'Pressure').sum())
    chain_under_pres = int(chain['under_pressure'].fillna(False).sum())
    chain_dribbles = int((chain['type'] == 'Dribble').sum())

    passes_chain = chain[chain['type'] == 'Pass']

    chain_has_cross = int(passes_chain['pass_cross'].fillna(False).any())
    chain_has_throughball = int(passes_chain['pass_through_ball'].fillna(False).any())
    chain_has_switchplay = int(passes_chain['pass_switch'].fillna(False).any())

    first_loc = chain.iloc[0]['location'] if len(chain) > 0 else None
    chain_start_x = float(first_loc[0]) if isinstance(first_loc, list) else 60.0
    chain_prog_distance = float(shot_loc[0] - chain_start_x)

    fast_break = int(chain_duration < 4.0 and w_speed > 3.0)

    def pass_end_in_box(row):
        try:
            end = row['pass_end_location']
            return isinstance(end, list) and end[0] > 102 and 18 < end[1] < 62
        except Exception:
            return False

    pass_into_box = int(passes_chain.apply(pass_end_in_box, axis=1).any())

    carries_chain = chain[chain['type'] == 'Carry']

    def carry_end_in_box(row):
        try:
            end = row['carry_end_location']
            return isinstance(end, list) and end[0] > 102 and 18 < end[1] < 62
        except Exception:
            return False

    carry_into_box = int(carries_chain.apply(carry_end_in_box, axis=1).any())

    dribbles_chain = chain[chain['type'] == 'Dribble']
    n_defenders_beaten = int(dribbles_chain['dribble_outcome'].fillna('').eq('Complete').sum())

    final_pass_length = 0.0
    final_pass_angle = 0.0
    if len(passes_chain) > 0:
        last_pass = passes_chain.iloc[-1]
        final_pass_length = float(last_pass['pass_length']
                                   if pd.notna(last_pass.get('pass_length')) else 0.0)
        final_pass_angle = float(last_pass['pass_angle']
                                  if pd.notna(last_pass.get('pass_angle')) else 0.0)

    prev_events = match_events[
        (match_events['period'] == shot_period) &
        (match_events['time_seconds'] < shot_time)
    ].sort_values('time_seconds').tail(3)
    shot_from_rebound = int(prev_events['type'].isin(['Block', 'Goal Keeper', 'Clearance']).any())

    freeze = shot_row.get('shot_freeze_frame')
    defensive_line_x = 0.0
    if isinstance(freeze, list) and len(freeze) > 0:
        opponents = [p for p in freeze if not p.get('teammate') and not is_goalkeeper(p)]
        if opponents:
            defensive_line_x = float(np.mean([p['location'][0] for p in opponents]))

    return pd.Series({
        'w_passes': w_passes,
        'w_carries': w_carries,
        'w_pressure_events': w_pressure_events,
        'w_miscontrol': w_miscontrol,
        'w_dribble': w_dribble,
        'w_under_pressure': w_under_pressure,
        'w_prog_distance': w_prog_distance,
        'w_speed': w_speed,
        'chain_passes': chain_passes,
        'chain_carries': chain_carries,
        'chain_duration': chain_duration,
        'chain_pressures': chain_pressures,
        'chain_under_pressure': chain_under_pres,
        'chain_dribbles': chain_dribbles,
        'chain_has_cross': chain_has_cross,
        'chain_has_throughball': chain_has_throughball,
        'chain_has_switchplay': chain_has_switchplay,
        'chain_start_x': chain_start_x,
        'chain_prog_distance': chain_prog_distance,
        'fast_break': fast_break,
        'pass_into_box': pass_into_box,
        'carry_into_box': carry_into_box,
        'n_defenders_beaten': n_defenders_beaten,
        'final_pass_length': final_pass_length,
        'final_pass_angle': final_pass_angle,
        'shot_from_rebound': shot_from_rebound,
        'defensive_line_x': defensive_line_x,
    })


zero_row = {
    'w_passes': 0, 'w_carries': 0, 'w_pressure_events': 0,
    'w_miscontrol': 0, 'w_dribble': 0, 'w_under_pressure': 0,
    'w_prog_distance': 0.0, 'w_speed': 0.0,
    'chain_passes': 0, 'chain_carries': 0, 'chain_duration': 0.0,
    'chain_pressures': 0, 'chain_under_pressure': 0, 'chain_dribbles': 0,
    'chain_has_cross': 0, 'chain_has_throughball': 0,
    'chain_has_switchplay': 0, 'chain_start_x': 60.0,
    'chain_prog_distance': 0.0, 'fast_break': 0,
    'pass_into_box': 0, 'carry_into_box': 0, 'n_defenders_beaten': 0,
    'final_pass_length': 0.0, 'final_pass_angle': 0.0,
    'shot_from_rebound': 0, 'defensive_line_x': 0.0,
}

temporal_rows = []

for idx, shot in df_rich.iterrows():
    try:
        match_ev = events_df[events_df['match_id'] == shot['match_id']]
        feats = extract_temporal_features(shot, match_ev)
        feats['shot_id'] = shot['id']
        temporal_rows.append(feats)
    except Exception:
        row = zero_row.copy()
        row['shot_id'] = shot['id']
        temporal_rows.append(pd.Series(row))

df_temporal = pd.DataFrame(temporal_rows)

temporal_cols = [c for c in df_temporal.columns if c != 'shot_id']
for col in temporal_cols:
    df_rich[col] = df_temporal[col].values

print(f"Temporal features merged: {len(temporal_cols)}")
print(df_rich.groupby('is_goal')[
    ['w_speed', 'w_prog_distance', 'chain_has_throughball',
     'chain_prog_distance', 'chain_pressures', 'defensive_line_x']
].mean().round(3).T.rename(columns={0: 'Non-Goal', 1: 'Goal'}))

# Checkpoint used in this thesis was built from StatsBomb open data accessed 01-07-2026.
# Rebuilding from scratch may give slightly different results due to StatsBomb data
# updates; the verified checkpoint (df_rich_with_temporal_v2.pkl) is provided in the repo.
df_rich.to_pickle('data/df_rich_temporal_v3.pkl')